# Shift-Equivariant Downsampling Strategies Analysis

## 🎯 **Research Objective**

This notebook provides a comprehensive evaluation of different downsampling strategies in Complex-Valued Neural Networks (CVNNs) for SAR/PolSAR data processing, with a focus on **shift-equivariance preservation**. 

### 🔬 **Scientific Context**

Traditional downsampling methods (MaxPool, AvgPool) break the shift-equivariance property that is crucial for accurate SAR data analysis. This research evaluates novel learnable downsampling strategies:

- **Traditional Methods**: MaxPool, AvgPool, Low-Pass Filter (LPF)
- **Learnable Projective Downsampling (LPD)**: With/without Fourier processing
- **Adaptive Projective Downsampling (APD)**: With/without Fourier processing  
- **Projection Functions**: Amplitude, Polynomial, MLP

### 🎯 **Research Questions**

1. **Performance**: How do different downsampling strategies affect task performance?
2. **Shift-Equivariance**: Which methods best preserve shift-equivariance properties?
3. **Computational Efficiency**: What are the parameter/computational trade-offs?
4. **Task Specificity**: Do optimal strategies differ between reconstruction and segmentation?
5. **Projection Impact**: How do different projection functions affect learnable methods?

### 📊 **Experimental Design**

- **Reconstruction Task**: ALOS dataset with AutoEncoder architecture
- **Segmentation Task**: PolSF dataset with UNet architecture  
- **Evaluation Metrics**: Standard task metrics + H/Alpha & Cameron classification (reconstruction only)
- **Shift Analysis**: Circular shift consistency automatically computed during evaluation
- **Statistical Rigor**: Multiple runs with confidence intervals

This analysis will serve as the foundation for the **"Shift-equivariant convolutional complex-valued neural networks"** research paper.

## Configuration & Setup

### 📋 **Experiment Configuration**

Define the experimental parameters and toggle different analysis components.

In [1]:
# Downsampling Impact Analysis Configuration
# Following the same pattern as preserving_polsar_properties notebook

import pathlib
import time
import logging

# Experiment control flags - what to evaluate
evaluate_traditional_methods = True     # MaxPool, AvgPool, LPF
evaluate_learnable_methods = True       # LPD, LPD_F, APD, APD_F  
evaluate_projection_functions = True    # Amplitude, Polynomial, MLP
evaluate_reconstruction = True          # ALOS dataset reconstruction
evaluate_segmentation = False            # PolSF dataset segmentation

# Analysis settings
enable_sensitivity_analysis = False      # Multiple runs for statistical rigor
sensitivity_n_runs = 2                # Number of runs per experiment
sensitivity_different_seeds = True      # Use different seeds for reliability

# Computational settings  
max_epochs_full = 50                    # Full training epochs
save_detailed_results = True            # Save comprehensive results
enable_wandb = True                    # Weights & Biases logging

# Results directory
results_dir = "logs/downsampling_impact_analysis"
pathlib.Path(results_dir).mkdir(parents=True, exist_ok=True)

# Reproducibility
fixed_seed = 42
sensitivity_base_seed = 42

# Replace print with logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

logger.info("🎯 Downsampling Impact Analysis - Configuration")
logger.info(f"   Traditional methods: {evaluate_traditional_methods}")  
logger.info(f"   Learnable methods: {evaluate_learnable_methods}")
logger.info(f"   Projection analysis: {evaluate_projection_functions}")
logger.info(f"   Reconstruction task: {evaluate_reconstruction}")
logger.info(f"   Segmentation task: {evaluate_segmentation}")
logger.info(f"   Sensitivity analysis: {enable_sensitivity_analysis} ({sensitivity_n_runs} runs)")
logger.info(f"   Results: {results_dir}")
logger.info(f"   Base seed: {fixed_seed}")

# Verify we can access existing config files
project_root = pathlib.Path.cwd()
configs_path = project_root / "configs"

reconstruction_config_path = configs_path / "config_reconstruction.yaml" 
segmentation_config_path = configs_path / "config_segmentation.yaml"

logger.info(f"\n📁 Configuration files:")
logger.info(f"   Reconstruction config: {reconstruction_config_path.exists()} - {reconstruction_config_path}")
logger.info(f"   Segmentation config: {segmentation_config_path.exists()} - {segmentation_config_path}")

if not reconstruction_config_path.exists() or not segmentation_config_path.exists():
    logger.error("❌ Required config files not found!")
    logger.error("   Please ensure you're running from the project root directory")
else:
    logger.info("✅ Configuration setup complete!")

🎯 Downsampling Impact Analysis - Configuration
   Traditional methods: True
   Learnable methods: True
   Projection analysis: True
   Reconstruction task: True
   Segmentation task: False
   Sensitivity analysis: False (2 runs)
   Results: logs/downsampling_impact_analysis
   Base seed: 42

📁 Configuration files:
   Reconstruction config: True - /home/qgabot/Documents/cvnn/notebooks/configs/config_reconstruction.yaml
   Segmentation config: True - /home/qgabot/Documents/cvnn/notebooks/configs/config_segmentation.yaml
✅ Configuration setup complete!


# Impact of Downsampling Layers on SAR Reconstruction and Segmentation Models

## 🎯 A Comprehensive Analysis for Deep Learning in SAR/PolSAR Processing

[![Paper](https://img.shields.io/badge/Paper-arXiv-red)](https://arxiv.org/abs/your-paper-id)
[![Code](https://img.shields.io/badge/Code-GitHub-green)](https://github.com/QuentinGABOT/cvnn)
[![License](https://img.shields.io/badge/License-MIT-blue)](LICENSE)

---

### 📖 Abstract

This notebook presents a systematic evaluation of downsampling layer strategies in deep learning models for Synthetic Aperture Radar (SAR) and Polarimetric SAR (PolSAR) data processing. We comprehensively analyze the impact of different downsampling approaches on:

- **🎯 Segmentation Models**: U-Net and SegNet architectures for pixel-wise classification
- **🔄 Reconstruction Models**: AutoEncoder and LatentAutoEncoder for image reconstruction
- **📊 Performance Metrics**: IoU, Dice, PSNR, SSIM, and computational efficiency
- **🧠 Feature Preservation**: Analysis of spatial information retention across different strategies

### 🔬 Research Questions

1. **How do different downsampling strategies affect model performance across segmentation and reconstruction tasks?**
2. **Which downsampling approaches best preserve SAR-specific features and polarimetric properties?**
3. **What are the computational trade-offs between different downsampling methods?**
4. **Can we identify task-specific optimal downsampling configurations?**
5. **How do learnable vs. fixed downsampling layers compare in terms of performance and efficiency?**

### 🏗️ Methodology Overview

Our analysis framework systematically evaluates multiple downsampling strategies:

- **Max Pooling**: Traditional feature extraction with different kernel sizes
- **Average Pooling**: Smooth downsampling preserving local averages  
- **Strided Convolution**: Learnable spatial reduction with parameter efficiency
- **Learnable Downsampling**: Custom adaptive downsampling layers
- **Hybrid Approaches**: Combinations of multiple downsampling strategies

Each strategy is evaluated across multiple model architectures with consistent experimental protocols, ensuring fair comparison and reproducible results.

### 📊 Key Contributions

- **Comprehensive Evaluation**: First systematic study of downsampling impact on SAR model architectures
- **Cross-Task Analysis**: Unified comparison framework for segmentation and reconstruction tasks
- **Statistical Rigor**: Multiple runs with confidence intervals and significance testing
- **Practical Guidelines**: Actionable recommendations for downsampling layer selection
- **Open Source**: Complete experimental code and data for reproducibility

---

## Environment Setup and Configuration

### 🛠️ Parameters and Experiment Configuration

Define notebook parameters for reproducibility and comprehensive downsampling analysis. These parameters control the scope and scale of our experiments across different model architectures and downsampling strategies.

In [2]:
# Import required libraries and set up project paths
import sys
import os
import copy
from pathlib import Path
import warnings
import subprocess
warnings.filterwarnings('ignore')

# Set up paths for cvnn package imports
current_dir = Path.cwd()
project_root = current_dir.parent  # Go up from notebooks/ to cvnn/

# Scientific computing libraries
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Any
import yaml
import json
from datetime import datetime
import pandas as pd

# Optional dependencies
try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False

# CVNN package imports
from cvnn.models.utils import get_downsampling, get_upsampling, get_projection
from cvnn.config import load_config, validate_config_consistency
from cvnn.config_utils import get_model_params, validate_required_config_sections
from cvnn.experiments import run_experiment
from cvnn.utils import setup_logging, set_seed

# Replace print with logging
logger.info(f"✅ All imports successful!")
logger.info(f"📁 Project root: {project_root}")
logger.info(f"📈 WANDB available: {WANDB_AVAILABLE}")

✅ All imports successful!
📁 Project root: /home/qgabot/Documents/cvnn
📈 WANDB available: True


In [3]:
# Set up reproducibility
set_seed(fixed_seed)
logger.info(f"🎯 Fixed seed set to: {fixed_seed}")

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_cuda = torch.cuda.is_available()
logger.info(f"🖥️  Using device: {device}")

# Create results directory
results_path = pathlib.Path(results_dir)
results_path.mkdir(parents=True, exist_ok=True)
logger.info(f"📁 Results will be saved to: {results_path}")

# Log environment info
logger.info(f"\n📋 Environment Info:")
logger.info(f"   Python: {sys.version.split()[0]}")
logger.info(f"   NumPy: {np.__version__}")
logger.info(f"   PyTorch: {torch.__version__}")

# Log Git commit if available
git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"]).strip().decode()
logger.info(f"   Git SHA: {git_sha[:8]}")

# Configure matplotlib for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

logger.info("\n✅ Environment setup complete!")

🎯 Fixed seed set to: 42
🖥️  Using device: cuda
📁 Results will be saved to: logs/downsampling_impact_analysis

📋 Environment Info:
   Python: 3.12.3
   NumPy: 2.2.6
   PyTorch: 2.7.0+cu126
   Git SHA: b8ed800a

✅ Environment setup complete!


## Downsampling Strategy Definition

### 🏗️ Systematic Downsampling Layer Configurations

This section defines the comprehensive set of downsampling strategies we will evaluate. Each strategy represents a different approach to spatial reduction, with varying impacts on:

- **Information Preservation**: How well spatial features are maintained
- **Computational Efficiency**: Parameter count and computational cost
- **Learning Capacity**: Ability to adapt to specific tasks
- **Receptive Field**: Impact on the model's spatial understanding

### 📊 Downsampling Strategy Categories

1. **Traditional Pooling**: MaxPool and AvgPool with different kernel sizes
2. **Low-Pass Filtering**: LPF with smooth spatial reduction
3. **Learnable Polyphase Decimation (LPD)**: Custom adaptive downsampling with sophisticated projection functions
4. **Amplitude-Phase Decimation (APD)**: Parameter-efficient downsampling with amplitude-phase awareness
5. **Fourier Variants**: LPD_F and APD_F for frequency-domain processing

### 🎯 Projection Function Strategy

**Key Insight**: Different downsampling approaches require different projection strategies based on their internal mechanisms:

- **LPD Variants** (`LPD`, `LPD_F`): Require sophisticated projection functions (`amplitude`, `polynomial`, `MLP`) as they are integral to the downsampling scheme itself
- **APD Variants** (`APD`, `APD_F`): Use `amplitude` projection only - sufficient for supervised tasks with no learnable projection parameters
- **Traditional Methods** (`maxpool`, `avgpool`, `LPF`): Use `amplitude` projection only - standard approach for supervised classification/segmentation

### 🔄 Network Architecture Comparison

Additionally, we compare two fundamental network architectures:

- **Complex-CVNN**: Standard complex-valued neural networks (`layer_mode="complex"`)
- **Dual-RVNN**: Dual real-valued networks processing complex data (`layer_mode="real"` + `real_pipeline_type="complex_dual_real"`)

This comparison provides insights into the fundamental advantages of native complex-valued processing versus dual real-valued approaches.

In [4]:
# Load and inspect available downsampling strategies from the existing pipeline
print("🔍 Discovering available downsampling strategies from mode_utils...")

# Get available downsampling layers
available_downsampling = []
downsampling_types = ["maxpool", "avgpool", "LPD", "LPD_F", "APD", "APD_F", "LPF"]

for ds_type in downsampling_types:
    try:
        layer = get_downsampling(
            downsampling=ds_type,
            layer_mode="complex",
            factor=2,
            in_channels=2,
            out_channels=2
        )
        if layer is not None:
            available_downsampling.append(ds_type)
            print(f"✅ {ds_type}: Available")
        else:
            print(f"❌ {ds_type}: Not available")
    except Exception as e:
        print(f"❌ {ds_type}: Error - {str(e)[:50]}...")

print(f"\n📊 Found {len(available_downsampling)} available downsampling strategies:")
for ds in available_downsampling:
    print(f"   • {ds}")

# Get available upsampling layers (for paired strategies)
available_upsampling = []
upsampling_types = ["transpose", "bilinear", "nearest", "LPU", "LPU_F", "APU", "APU_F"]

for us_type in upsampling_types:
    try:
        # Replace print with logging
        logger.info(f"Evaluating upsampling: {us_type}")
        layer = get_upsampling(
            upsampling=us_type,
            layer_mode="complex",
            factor=2,
            in_channels=2,
            out_channels=2
        )
        if layer is not None:
            available_upsampling.append(us_type)
            print(f"✅ {us_type}: Available")
        else:
            print(f"❌ {us_type}: Not available")
    except Exception as e:
        print(f"❌ {us_type}: Error - {str(e)[:50]}...")

print(f"\n📈 Found {len(available_upsampling)} available upsampling strategies:")
for us in available_upsampling:
    print(f"   • {us}")

# Get available projection functions (for learnable methods)
available_projections = []
projection_types = ["amplitude", "polynomial", "MLP"]

for proj_type in projection_types:
    try:
        layer = get_projection(
            projection=proj_type,
            layer_mode="complex",
            projection_config={"input_channels": 2, "output_channels": 1}
        )
        if layer is not None:
            available_projections.append(proj_type)
            print(f"✅ {proj_type}: Available")
        else:
            print(f"❌ {proj_type}: Not available")
    except Exception as e:
        print(f"❌ {proj_type}: Error - {str(e)[:50]}...")

print(f"\n🧮 Found {len(available_projections)} available projection functions:")
for proj in available_projections:
    print(f"   • {proj}")

print("\n🎯 Pipeline discovery complete!")

🔍 Discovering available downsampling strategies from mode_utils...
✅ maxpool: Available
✅ avgpool: Available
✅ LPD: Available
✅ LPD_F: Available
✅ APD: Available
✅ APD_F: Available
✅ LPF: Available

📊 Found 7 available downsampling strategies:
   • maxpool
   • avgpool
   • LPD
   • LPD_F
   • APD
   • APD_F
   • LPF
✅ transpose: Available
✅ bilinear: Available
✅ nearest: Available
✅ LPU: Available
✅ LPU_F: Available
✅ APU: Available
✅ APU_F: Available

📈 Found 7 available upsampling strategies:
   • transpose
   • bilinear
   • nearest
   • LPU
   • LPU_F
   • APU
   • APU_F
✅ amplitude: Available
✅ polynomial: Available
✅ MLP: Available

🧮 Found 3 available projection functions:
   • amplitude
   • polynomial
   • MLP

🎯 Pipeline discovery complete!


### 🎯 Experiment Configuration Using Existing Pipeline with Comprehensive LPD Testing

Now that we've discovered all available downsampling strategies, let's configure our experiments using the existing pipeline infrastructure. We'll leverage the existing config files and systematically test different downsampling approaches with **comprehensive LPD evaluation** for complex layers only.

**Experiment Strategy:**
- **Base Configs**: Use `config_segmentation.yaml` (PolSF dataset) and `config_reconstruction.yaml` (ALOS dataset)
- **Systematic Testing**: Test all available downsampling/upsampling pairs
- **Complex Layer Comprehensive LPD Testing**: 
  - **Projection Layer Testing**: LPD variants test `amplitude`, `polynomial`, `MLP` projections (traditional approach)
  - **Gumbel Softmax Testing**: LPD variants test `"mean"` and `"product"` Gumbel Softmax types with `projection_layer=None`
  - **Non-LPD variants**: Use `projection_layer=None` (safe case automatically adds amplitude projection)
- **Real Layer Simple Testing**:
  - **No Projection/Gumbel Softmax**: Real layers process real-valued data directly, no complex-to-real conversion needed
  - **Standard Downsampling**: Test all downsampling strategies with standard real-valued processing
- **Architecture Comparison**: Complex-CVNN vs Dual-RVNN with appropriate testing for each
- **Statistical Rigor**: Multiple runs with different seeds for confidence intervals

**Key Distinction - Layer Mode Specific Testing:**
- **Complex Layers (`layer_mode="complex"`)**: Require projection or Gumbel Softmax for complex-to-real conversion in supervised tasks
- **Real Layers (`layer_mode="real"`)**: Process real-valued data directly, no conversion needed

**Total Experiments**: ~60 configurations 
- **Complex-CVNN**: ~50 configurations (comprehensive LPD testing + non-LPD variants)
- **Dual-RVNN**: ~10 configurations (simple downsampling strategies only)

**Per Task Breakdown:**
- **Complex-CVNN**: 
  - LPD projection tests: 3 projections × 2 LPD variants = 6 configs
  - LPD Gumbel Softmax tests: 2 types × 2 LPD variants = 4 configs  
  - Non-LPD variants: 8 traditional/APD configs
  - **Total**: 18 configurations per task
- **Dual-RVNN**: 
  - All downsampling strategies: 10 configs (no projection/Gumbel testing needed)
  - **Total**: 10 configurations per task

In [ ]:
# Define experiment configurations using existing pipeline with comprehensive LPD testing
print("🎯 Creating experiment configurations with comprehensive LPD testing...")

# Load base configurations
segmentation_config = load_config(segmentation_config_path)
reconstruction_config = load_config(reconstruction_config_path)

print(f"✅ Loaded segmentation config: {segmentation_config_path.name}")
print(f"✅ Loaded reconstruction config: {reconstruction_config_path.name}")

# Define paired downsampling/upsampling strategies to test
downsampling_pairs = [
    # Traditional methods
    #("maxpool", "transpose"),
    #("avgpool", "transpose"),
    #("avgpool", "bilinear"),
    #("avgpool", "nearest"),
    
    # Learnable Polyphase Decimation (LPD) variants - will test with projection layers AND Gumbel Softmax
    ("LPD", "LPU"),           # Standard learnable polyphase pair
    #("LPD_F", "LPU_F"),       # Fourier-domain variants
    
    # Amplitude-Phase Decimation (APD) variants - no learnable parameters
    ("APD", "APU"),           # Amplitude-phase aware pair
    #("APD_F", "APU_F"),       # Fourier-domain amplitude-phase pair
    
    # Low-pass filter method
    #("LPF", "transpose"),     # Low-pass filter with transpose upsampling
    ("LPF", "bilinear"),      # Low-pass filter with bilinear upsampling
]

# Define projection functions to test with LPD variants
projection_functions = ["amplitude", "polynomial", "MLP"]

# Define Gumbel Softmax types to test with LPD variants (when projection_layer=None)
#gumbel_softmax_types = ["mean", "product"]
gumbel_softmax_types = ["mean"]

# Create experiment configurations
experiment_configs = []

def create_task_experiments(base_config, task_name, layer_mode="complex"):
    """Create experiments for a specific task with optional layer mode override."""
    task_experiments = {}
    
    for ds, us in downsampling_pairs:
        # Base configuration
        config_name = f"{task_name}_{ds}_{us}"
        config_name += f"_{layer_mode}"
            
        config = copy.deepcopy(base_config)
        config["model"]["downsampling_layer"] = ds
        config["model"]["upsampling_layer"] = us
        config["model"]["layer_mode"] = layer_mode
        
        # Set real_pipeline_type for dual-RVNN experiments
        if layer_mode == "real":
            config["data"]["real_pipeline_type"] = "complex_dual_real"
        
        if layer_mode == "complex" and ds in ["LPD", "LPD_F"]:
            # LPD variants with complex layers: test both projection layers AND Gumbel Softmax types
            # 1. Test traditional projection layers (amplitude, polynomial, MLP)
            for proj in projection_functions:
                proj_config_name = f"{config_name}_{proj}"
                proj_config = copy.deepcopy(config)
                proj_config["model"]["projection_layer"] = proj
                if proj == "MLP":
                    proj_config["model"]["projection"] = {
                        "hidden_sizes": [8, 16, 8]
                    }
                elif proj == "polynomial":
                    proj_config["model"]["projection"] = {
                        "order": 5
                    }
                task_experiments[proj_config_name] = proj_config
            
            # 2. Test Gumbel Softmax types with projection_layer=None (safe case handles projection)
            for gumbel_type in gumbel_softmax_types:
                gumbel_config_name = f"{config_name}_gumbel_{gumbel_type}"
                gumbel_config = copy.deepcopy(config)
                gumbel_config["model"]["gumbel_softmax"] = gumbel_type
                task_experiments[gumbel_config_name] = gumbel_config
                
        else:            
            task_experiments[config_name] = config
    
    return task_experiments

if evaluate_segmentation:
    # Configure segmentation experiments (complex-CVNN)
    print("🎯 Creating segmentation experiments (complex-CVNN)...")
    segmentation_experiments = create_task_experiments(segmentation_config, "segmentation", "complex")
    experiment_configs.append(segmentation_experiments)

    # Configure segmentation experiments (dual-RVNN comparison)
    print("🎯 Creating segmentation experiments (dual-RVNN)...")
    segmentation_real_experiments = create_task_experiments(segmentation_config, "segmentation", "real")
    experiment_configs.append(segmentation_real_experiments)

if evaluate_reconstruction:
    # Configure reconstruction experiments (complex-CVNN)
    print("🎯 Creating reconstruction experiments (complex-CVNN)...")
    reconstruction_experiments = create_task_experiments(reconstruction_config, "reconstruction", "complex")
    experiment_configs.append(reconstruction_experiments)

    # Configure reconstruction experiments (dual-RVNN comparison)
    print("🎯 Creating reconstruction experiments (dual-RVNN)...")
    reconstruction_real_experiments = create_task_experiments(reconstruction_config, "reconstruction", "real")
    experiment_configs.append(reconstruction_real_experiments)

print(f"\n📋 Created {len(experiment_configs)} experiment configurations:")
if evaluate_segmentation:
    print(f"   • Segmentation (complex-CVNN): {len(segmentation_experiments)}")
    print(f"   • Segmentation (dual-RVNN): {len(segmentation_real_experiments)}")
if evaluate_reconstruction:
    print(f"   • Reconstruction (complex-CVNN): {len(reconstruction_experiments)}")
    print(f"   • Reconstruction (dual-RVNN): {len(reconstruction_real_experiments)}")

complex_lpd = 0
real_lpd = 0
complex_apd = 0
real_apd = 0
complex_traditional_experiments = 0
real_traditional_experiments = 0
total_complex_experiments = 0
total_real_experiments = 0

for batch_experiments in experiment_configs:
    for config_name, config in batch_experiments.items():
        if "LPD" in config_name:
            if config["model"]["layer_mode"] == "complex":
                complex_lpd += 1
            else:
                real_lpd += 1
        elif "APD" in config_name:
            if config["model"]["layer_mode"] == "complex":
                complex_apd += 1
            else:
                real_apd += 1
        elif "maxpool" in config_name or "avgpool" in config_name or "LPF" in config_name:
            if config["model"]["layer_mode"] == "complex":
                complex_traditional_experiments += 1
            else:
                real_traditional_experiments += 1
        
        if config["model"]["layer_mode"] == "complex":
            total_complex_experiments += 1
        else:
            total_real_experiments += 1

print(f"\n🔍 Experiment breakdown:")
print(f"   • CVNN LPD variants: {complex_lpd}")
print(f"   • RVNN LPD variants: {real_lpd}")
print(f"   • CVNN APD variants: {complex_apd}")
print(f"   • RVNN APD variants: {real_apd}")
print(f"   • CVNN traditional methods: {complex_traditional_experiments}")
print(f"   • RVNN traditional methods: {real_traditional_experiments}")
print(f"   • Total CVNN experiments: {total_complex_experiments}")
print(f"   • Total RVNN experiments: {total_real_experiments}")
print(f"   • Total experiments: {len(experiment_configs)}")


# Adjust experiment settings based on demo mode
for batch_experiments in experiment_configs:
    for config_name, config in batch_experiments.items():
        config["nepochs"] = max_epochs_full
        
        # Add experiment tracking
        if enable_wandb and WANDB_AVAILABLE:
            if "logging" not in config:
                config["logging"] = {}
            if "wandb" not in config["logging"]:
                config["logging"]["wandb"] = {}
            
            # Add layer_mode, projection, and gumbel_softmax_type to tags for better organization
            layer_mode = config["model"]["layer_mode"]
            task_type = config_name.split("_")[0]
            tags = ["downsampling", "comparison", task_type, f"{layer_mode}-mode"]
            
            # Add projection type tag if present
            if config["model"].get("projection_layer"):
                proj_type = config["model"]["projection_layer"]
                tags.append(f"proj-{proj_type}")
            
            # Add Gumbel Softmax type tag if present
            if "gumbel_softmax_type" in config["model"]:
                gumbel_type = config["model"]["gumbel_softmax_type"]
                tags.append(f"gumbel-{gumbel_type}")
            
            config["logging"]["wandb"].update({
                "project": "downsampling-impact-analysis",
                "name": config_name,
                "tags": tags
            })

print(f"\n⚙️  Experiment settings:")
print(f"   • Max epochs: {max_epochs_full}")
print(f"   • WANDB enabled: {enable_wandb and WANDB_AVAILABLE}")

print(f"\n💡 Corrected configuration strategy:")
print(f"   • Complex-CVNN LPD variants: Test projection layers AND Gumbel Softmax (complex-to-real conversion needed)")
print(f"   • Complex-CVNN non-LPD variants: Use projection_layer=None (safe case adds amplitude projection)")
print(f"   • Dual-RVNN all variants: No projection/Gumbel Softmax (real layers process real data directly)")

print(f"\n🎯 Complex Layer LPD comprehensive testing matrix:")
print(f"   • Projection layer tests: 3 projections × 2 LPD variants = 6 configs per task")
print(f"   • Gumbel Softmax tests: 2 types × 2 LPD variants = 4 configs per task")
print(f"   • Total complex LPD tests: 10 configs per task")

print(f"\n🔄 Network architecture comparison:")
print(f"   • Complex-CVNN: layer_mode='complex' (needs complex-to-real conversion for supervised tasks)")
print(f"   • Dual-RVNN: layer_mode='real' + real_pipeline_type='complex_dual_real' (processes real data directly)")

print("\n✅ Corrected experiment configurations ready!")

## Running All Experiments

Execute all defined experiments systematically. Each experiment will be run independently with its own configuration, and results will be collected for analysis.

In [ ]:
from cvnn.experiments import run_experiment
import tempfile
import os
import numpy as np
from scipy import stats

def run_pipeline_experiment(exp_name: str, exp_config: Dict[str, Any], exp_index: int, total_experiments: int, run_id: int = 0, seed_offset: int = 0) -> Dict[str, Any]:
    """
    Run a single experiment using the standard reconstruction pipeline.
    
    This function uses the project's standard run_experiment function with configuration variants,
    ensuring consistency with the main project workflow.
    
    Args:
        exp_config: Experiment configuration dictionary
        exp_index: Current experiment index (0-based)
        total_experiments: Total number of experiments
        run_id: Run identifier for sensitivity analysis (0-based)
        seed_offset: Offset to add to the base seed for this run
        
    Returns:
        Dictionary with experiment results and metadata
    """
    
    if enable_sensitivity_analysis and sensitivity_n_runs > 1:
        print(f"\n🧪 Experiment {exp_index + 1}/{total_experiments} - Run {run_id + 1}/{sensitivity_n_runs}: {exp_name}")
    else:
        print(f"\n🧪 Experiment {exp_index + 1}/{total_experiments}: {exp_name}")
    print(f"   Started at: {datetime.now().strftime('%H:%M:%S')}")
    if seed_offset > 0:
        print(f"   Random seed: {fixed_seed + seed_offset}")
    
    start_time = time.time()
        
    # Add experiment metadata
    exp_config["experiment"] = {
        "name": exp_name,
        "index": exp_index,
        "run_id": run_id,
        "seed_offset": seed_offset
    }
    
    # Set random seed for this run if sensitivity analysis is enabled
    if enable_sensitivity_analysis and sensitivity_different_seeds:
        exp_config["seed"] = fixed_seed + seed_offset
    else:
        exp_config["seed"] = fixed_seed
    
    print(f"   ⚙️  Configuration created with task: {exp_config.get('task', 'unknown')}")
    
    # 2. Create temporary config file for the pipeline
    with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
        yaml.dump(exp_config, f, default_flow_style=False)
        temp_config_path = f.name
    
    print(f"   📝 Created temporary config: {temp_config_path}")
    
    # 3. Run the experiment using the standard pipeline
    print(f"   🚀 Running experiment via standard pipeline...")

    try:
        # Use the project's standard run_experiment function
        history, metrics, logdir, model = run_experiment(
            config_path=temp_config_path,
            resume_logdir=False,
            mode_override=None
        )
        
        print(f"   ✅ Pipeline execution completed")
        print(f"   📁 Results saved to: {logdir}")
        
        # Extract training history
        training_history = {}
        if history:
            if isinstance(history, dict):
                training_history = history
            else:
                # Convert history object to dict if needed
                training_history = {
                    "train_loss": getattr(history, 'train_loss', []),
                    "valid_loss": getattr(history, 'valid_loss', []),
                }
        
        # Calculate final losses
        final_train_loss = training_history.get("train_loss", [])[-1] if training_history.get("train_loss") else None
        final_val_loss = training_history.get("valid_loss", [])[-1] if training_history.get("valid_loss") else None
        best_val_loss = min(training_history.get("valid_loss", [])) if training_history.get("valid_loss") else None
        best_epoch = training_history.get("valid_loss", []).index(best_val_loss) if best_val_loss else None
        
        print(f"   📊 Final validation loss: {final_val_loss:.6f}" if final_val_loss else "   📊 No validation loss recorded")
        print(f"   📈 Evaluation metrics: {metrics}")
        
    except Exception as pipeline_error:
        print(f"   ❌ Pipeline execution failed: {pipeline_error}")
        raise pipeline_error
    
    finally:
        # Clean up temporary config file
        if os.path.exists(temp_config_path):
            os.unlink(temp_config_path)
    
    # 4. Compile results in standard format
    end_time = time.time()
    duration = end_time - start_time
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    pipeline_results = {
        # Experiment metadata
        "experiment_name": exp_name,
        "experiment_index": exp_index,
        "run_id": run_id,
        "seed_offset": seed_offset,
        "duration_seconds": duration,
        "timestamp": datetime.now().isoformat(),
        
        # Configuration
        "config": config,
        "layer_mode": config.get("model", {}).get("layer_mode", "unknown"),
        "model_class": config.get("model", {}).get("class", "AutoEncoder"),
        
        # Model info
        "total_parameters": total_params,
        "trainable_parameters": trainable_params,
        
        # Training results
        "final_train_loss": final_train_loss,
        "final_val_loss": final_val_loss,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "training_history": training_history,
        
        # Evaluation metrics
        "metrics": metrics,
        
        # Pipeline info
        "logdir": str(logdir),
        
        # Status
        "status": "completed",
        "error": None
    }
    
    print(f"   ✅ Experiment completed in {duration:.1f}s")
    
    return pipeline_results


def run_sensitivity_experiment(exp_name: str, exp_config: Dict[str, Any], exp_index: int, total_experiments: int) -> List[Dict[str, Any]]:
    """
    Run an experiment multiple times for sensitivity analysis.
    
    Args:
        exp_config: Experiment configuration dictionary
        exp_index: Current experiment index (0-based)
        total_experiments: Total number of experiments
        
    Returns:
        List of results from all runs
    """
    if not enable_sensitivity_analysis or sensitivity_n_runs <= 1:
        # Run single experiment
        return [run_pipeline_experiment(exp_name, exp_config, exp_index, total_experiments)]
    
    print(f"\n🔄 Sensitivity Analysis: Running {sensitivity_n_runs} times for {exp_name}")
    
    all_runs = []
    
    for run_id in range(sensitivity_n_runs):
        # Calculate seed offset for this run
        seed_offset = run_id * 1000 if sensitivity_different_seeds else 0
        result = run_pipeline_experiment(exp_name, exp_config, exp_index, total_experiments, run_id, seed_offset)
        all_runs.append(result)
    
    # Print summary for this experiment
    successful_runs = [r for r in all_runs if r.get("status") == "completed"]
    if len(successful_runs) > 0:
        val_losses = [r["final_val_loss"] for r in successful_runs if r.get("final_val_loss") is not None]
        if val_losses:
            mean_loss = np.mean(val_losses)
            std_loss = np.std(val_losses)
            print(f"   📊 Sensitivity Summary: {len(successful_runs)}/{sensitivity_n_runs} successful runs")
            print(f"   📈 Validation Loss: {mean_loss:.6f} ± {std_loss:.6f}")
    
    return all_runs


def calculate_sensitivity_statistics(results: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Calculate statistical measures across multiple runs of the same experiment.
    
    Args:
        results: List of results from multiple runs of the same experiment
        
    Returns:
        Dictionary with statistical measures
    """
    successful_results = [r for r in results if r.get("status") == "completed"]
    
    if len(successful_results) == 0:
        return {
            "n_successful_runs": 0,
            "n_failed_runs": len(results),
            "success_rate": 0.0
        }
    
    stats_dict = {
        "n_successful_runs": len(successful_results),
        "n_failed_runs": len(results) - len(successful_results),
        "success_rate": len(successful_results) / len(results),
        "experiment_name": successful_results[0]["experiment_name"],
    }
    
    # Calculate statistics for numerical metrics
    numerical_fields = [
        "final_train_loss", "final_val_loss", "best_val_loss", "best_epoch",
        "total_parameters", "trainable_parameters", "duration_seconds"
    ]
    
    for field in numerical_fields:
        values = [r[field] for r in successful_results if r.get(field) is not None]
        if values:
            stats_dict[f"{field}_mean"] = np.mean(values)
            stats_dict[f"{field}_std"] = np.std(values)
            stats_dict[f"{field}_min"] = np.min(values)
            stats_dict[f"{field}_max"] = np.max(values)
            if len(values) > 1:
                # Calculate confidence interval (95%)
                confidence_level = 0.95
                degrees_freedom = len(values) - 1
                sample_mean = np.mean(values)
                sample_stderr = stats.sem(values)
                confidence_interval = stats.t.interval(confidence_level, degrees_freedom, sample_mean, sample_stderr)
                stats_dict[f"{field}_ci_lower"] = confidence_interval[0]
                stats_dict[f"{field}_ci_upper"] = confidence_interval[1]
    
    # Calculate statistics for metrics
    if len(successful_results) > 0 and "metrics" in successful_results[0]:
        
        def extract_metric_values(metric_dict, prefix=""):
            """Recursively extract metric values, flattening nested dictionaries."""
            values = {}
            for key, value in metric_dict.items():
                full_key = f"{prefix}_{key}" if prefix else key
                if isinstance(value, dict):
                    values.update(extract_metric_values(value, full_key))
                elif isinstance(value, (int, float)) and not isinstance(value, bool):
                    values[full_key] = value
            return values
        
        # Get all metric keys from first result
        all_metric_keys = set()
        for result in successful_results:
            if "metrics" in result:
                metric_values = extract_metric_values(result["metrics"])
                all_metric_keys.update(metric_values.keys())

        # Calculate statistics for each metric
        for metric_key in all_metric_keys:
            values = []
            for result in successful_results:
                if "metrics" in result:
                    metric_values = extract_metric_values(result["metrics"])
                    if metric_key in metric_values:
                        values.append(metric_values[metric_key])
            
            if values:
                stats_dict[f"metric_{metric_key}_mean"] = np.mean(values)
                stats_dict[f"metric_{metric_key}_std"] = np.std(values)
                stats_dict[f"metric_{metric_key}_min"] = np.min(values)
                stats_dict[f"metric_{metric_key}_max"] = np.max(values)
                if len(values) > 1:
                    confidence_level = 0.95
                    degrees_freedom = len(values) - 1
                    sample_mean = np.mean(values)
                    sample_stderr = stats.sem(values)
                    confidence_interval = stats.t.interval(confidence_level, degrees_freedom, sample_mean, sample_stderr)
                    stats_dict[f"metric_{metric_key}_ci_lower"] = confidence_interval[0]
                    stats_dict[f"metric_{metric_key}_ci_upper"] = confidence_interval[1]
    
    # Copy configuration details from first successful result
    if len(successful_results) > 0:
        first_result = successful_results[0]
        for key in ["layer_mode", "model_class"]:
            if key in first_result:
                stats_dict[key] = first_result[key]
        
        # Extract config details
        if "config" in first_result:
            config = first_result["config"]
            model_config = config.get("model", {})
            stats_dict["latent_dim"] = model_config.get("latent_dim", np.nan)
            stats_dict["num_layers"] = model_config.get("num_layers", np.nan)
            stats_dict["channels_ratio"] = model_config.get("channels_ratio", np.nan)
    
    return stats_dict

def run_batch_experiments(batch_experiments: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Run all experiments in the provided configurations.
    
    Args:
        batch_experiments: Dictionnary of experiments name and configuration
    Returns:
        List of results from all experiments
    """
    # Initialize results collection
    all_results = []  # Individual run results
    sensitivity_stats = []  # Aggregated statistics per experiment
    experiment_start_time = time.time()

    print(f"🚀 Starting pipeline evaluation with {len(batch_experiments)} experiments")
    if enable_sensitivity_analysis:
        total_runs = len(batch_experiments) * sensitivity_n_runs
        print(f"   🔄 Sensitivity Analysis: {sensitivity_n_runs} runs per experiment ({total_runs} total runs)")
        print(f"   🎲 Different seeds: {sensitivity_different_seeds}")
    else:
        print(f"   🔄 Single run mode (sensitivity analysis disabled)")
    print(f"   Max epochs: {max_epochs_full}")
    print(f"   Device: {device}")
    print(f"   Results dir: {results_dir}")

    i = 0
    # Run all experiments using the sensitivity analysis approach
    for exp_name, exp_config in batch_experiments.items():
        # Run the experiment (potentially multiple times for sensitivity analysis)
        print(f"\n🔄 Running experiment {i + 1}/{len(batch_experiments)}: {exp_name}")
        experiment_runs = run_sensitivity_experiment(exp_name, exp_config, i, len(batch_experiments))
        
        # Add all individual runs to results
        all_results.extend(experiment_runs)
        
        # Calculate sensitivity statistics
        if enable_sensitivity_analysis and len(experiment_runs) > 1:
            experiment_stats = calculate_sensitivity_statistics(experiment_runs)
            sensitivity_stats.append(experiment_stats)
            
            # Print sensitivity summary
            if experiment_stats["n_successful_runs"] > 0:
                print(f"\n📊 Sensitivity Analysis Summary for {exp_name}:")
                print(f"   Success Rate: {experiment_stats['success_rate']:.1%} ({experiment_stats['n_successful_runs']}/{len(experiment_runs)})")
                
                # Print key metrics with confidence intervals
                key_metrics = [
                    ("final_val_loss", "Validation Loss"),
                    ("best_val_loss", "Best Validation Loss"),
                    ("duration_seconds", "Training Time (s)"),
                ]
                
                for metric_key, metric_name in key_metrics:
                    if f"{metric_key}_mean" in experiment_stats:
                        mean_val = experiment_stats[f"{metric_key}_mean"]
                        std_val = experiment_stats[f"{metric_key}_std"]
                        ci_lower = experiment_stats.get(f"{metric_key}_ci_lower", None)
                        ci_upper = experiment_stats.get(f"{metric_key}_ci_upper", None)
                        
                        if ci_lower is not None and ci_upper is not None:
                            print(f"   {metric_name}: {mean_val:.6f} ± {std_val:.6f} (95% CI: [{ci_lower:.6f}, {ci_upper:.6f}])")
                        else:
                            print(f"   {metric_name}: {mean_val:.6f} ± {std_val:.6f}")
        else:
            # For single runs, just add the single result as "stats"
            if len(experiment_runs) > 0 and experiment_runs[0].get("status") == "completed":
                single_run_stats = {
                    "n_successful_runs": 1,
                    "n_failed_runs": 0,
                    "success_rate": 1.0,
                    "experiment_name": experiment_runs[0]["experiment_name"],
                }
                
                # Copy single run values as "mean" values
                for key, value in experiment_runs[0].items():
                    if isinstance(value, (int, float)) and not isinstance(value, bool):
                        single_run_stats[f"{key}_mean"] = value
                    elif key in ["layer_mode", "model_class", "latent_dim", "num_layers", "channels_ratio"]:
                        single_run_stats[key] = value
                
                sensitivity_stats.append(single_run_stats)
        
        # Save individual experiment runs if requested
        if save_detailed_results:
            for run_idx, run_result in enumerate(experiment_runs):
                if enable_sensitivity_analysis and sensitivity_n_runs > 1:
                    result_file = results_path / f"experiment_{i+1:02d}_{exp_name}_run_{run_idx+1}.json"
                else:
                    result_file = results_path / f"experiment_{i+1:02d}_{exp_name}.json"
                
                with open(result_file, 'w') as f:
                    # Convert numpy arrays to lists for JSON serialization
                    json_result = copy.deepcopy(run_result)
                    if 'training_history' in json_result and json_result['training_history']:
                        for key, values in json_result['training_history'].items():
                            if isinstance(values, np.ndarray):
                                json_result['training_history'][key] = values.tolist()
                    json.dump(json_result, f, indent=2, default=str)
            
            # Save sensitivity statistics
            if enable_sensitivity_analysis and len(experiment_runs) > 1:
                stats_file = results_path / f"experiment_{i+1:02d}_{exp_name}_sensitivity_stats.json"
                with open(stats_file, 'w') as f:
                    json.dump(experiment_stats, f, indent=2, default=str)
                print(f"   💾 Sensitivity statistics saved to: {stats_file}")
        i += 1

    # Calculate total time
    total_time = time.time() - experiment_start_time
    print(f"\n" + "="*80)
    print(f"🎉 All experiments completed!")
    print(f"   Total time: {total_time:.1f}s ({total_time/60:.1f} minutes)")

    if enable_sensitivity_analysis:
        total_individual_runs = len(all_results)
        print(f"   Total individual runs: {total_individual_runs}")
        print(f"   Average time per run: {total_time/total_individual_runs:.1f}s")
    else:
        print(f"   Average time per experiment: {total_time/len(experiment_configs):.1f}s")

    # Count successful vs failed experiments and runs
    successful_runs = sum(1 for r in all_results if r.get("status") == "completed")
    failed_runs = len(all_results) - successful_runs
    successful_experiments = sum(1 for s in sensitivity_stats if s.get("success_rate", 0) > 0)

    print(f"   Successful experiments: {successful_experiments}/{len(experiment_configs)}")
    print(f"   Successful runs: {successful_runs}/{len(all_results)}")

    if failed_runs > 0:
        print(f"   Failed runs: {failed_runs}/{len(all_results)}")
        print("   Failed runs:")
        for r in all_results:
            if r.get("status") == "failed":
                exp_name = r.get('experiment_name', 'unknown')
                run_id = r.get('run_id', 0)
                error = r.get('error', 'unknown error')
                print(f"     - {exp_name} (run {run_id + 1}): {error}")

    if enable_sensitivity_analysis:
        print(f"\n✅ Pipeline-based experiment execution with sensitivity analysis completed!")
        print(f"🔄 Each experiment was run {sensitivity_n_runs} times for statistical analysis")
    else:
        print(f"\n✅ Pipeline-based experiment execution completed!")
    return all_results, sensitivity_stats



print("✅ Pipeline-based experiment runner with sensitivity analysis defined")
print("🔄 Ready to execute experiments with statistical analysis across multiple runs")

## Results Aggregation and Analysis

Aggregate and analyze the results from all experiments to understand the impact of different pipeline configurations on reconstruction performance.

In [ ]:
# Create a comprehensive results DataFrame for analysis
def create_results_dataframe(results: List[Dict]) -> pd.DataFrame:
    """Create a pandas DataFrame from experiment results for analysis."""
    
    rows = []
    for result in results:
            
        # Extract basic info - use .get() to handle missing keys gracefully
        row = {
            "experiment_name": result.get("experiment_name", "unknown"),
            "description": result.get("experiment_description", "unknown"),
            "duration_seconds": result.get("duration_seconds", np.nan),
            "layer_mode": result.get("layer_mode", "unknown"),
            "model_class": result.get("model_class", "unknown"),
            "total_parameters": result.get("total_parameters", 0),
            "trainable_parameters": result.get("trainable_parameters", 0),
            "final_train_loss": result.get("final_train_loss", np.nan),
            "final_val_loss": result.get("final_val_loss", np.nan),
            "best_val_loss": result.get("best_val_loss", np.nan),
            "best_epoch": result.get("best_epoch", -1),
        }
        # Extract metrics with proper filtering
        if "metrics" in result:
            for metric_name, value in result["metrics"].items():
                if isinstance(value, dict):
                    # Flatten nested metrics
                    for sub_metric, sub_value in value.items():
                        if isinstance(sub_value, dict):
                            print(f"   Warning: Nested metric found for {metric_name}.{sub_metric}, skipping")
                        else:
                            # Skip complex metrics that shouldn't be in DataFrame
                            if any(substring in sub_metric for substring in ["confusion_matrix", "class_labels"]):
                                print(f"   Warning: Skipping complex metric {metric_name}.{sub_metric} for DataFrame")
                            else:
                                row[f"metric_{metric_name}_{sub_metric}"] = sub_value
                    # Also add the parent metric if it's not a dict
                    if not isinstance(value, dict):
                        row[f"metric_{metric_name}"] = value
                else:
                    row[f"metric_{metric_name}"] = value
        # Extract configuration details
        config = result.get("config", {})
        
        # Model config details
        model_config = config.get("model", {})
        row["num_layers"] = model_config.get("num_layers", np.nan)
        row["channels_ratio"] = model_config.get("channels_ratio", np.nan)
        row["latent_dim"] = model_config.get("latent_dim", np.nan)
        
        # Data config details  
        data_config = config.get("data", {})
        row["real_pipeline_type"] = data_config.get("real_pipeline_type", "none")
        row["num_transforms"] = len(data_config.get("transforms", []))
        
        # Training config details
        train_config = config.get("training", {})
        row["epochs"] = train_config.get("epochs", np.nan)
        row["learning_rate"] = train_config.get("learning_rate", np.nan)
        
        rows.append(row)
    
    return pd.DataFrame(rows)

def create_sensitivity_dataframe(sensitivity_stats: List[Dict]) -> pd.DataFrame:
    """Create a pandas DataFrame from sensitivity analysis statistics."""
    
    if not sensitivity_stats:
        return pd.DataFrame()
    
    rows = []
    for stats in sensitivity_stats:
        # Extract basic experiment info - use .get() to handle missing keys
        row = {
            "experiment_name": stats.get("experiment_name", "unknown"),
            "description": stats.get("experiment_description", "unknown"),
            "n_successful_runs": stats.get("n_successful_runs", 0),
            "n_failed_runs": stats.get("n_failed_runs", 0),
            "success_rate": stats.get("success_rate", 0.0),
            "layer_mode": stats.get("layer_mode", "unknown"),
            "model_class": stats.get("model_class", "unknown"),
        }
        
        # Extract mean, std, and confidence intervals for key metrics
        key_metrics = [
            "final_train_loss", "final_val_loss", "best_val_loss", "best_epoch",
            "total_parameters", "trainable_parameters", "duration_seconds"
        ]
        
        for metric in key_metrics:
            row[f"{metric}_mean"] = stats.get(f"{metric}_mean", np.nan)
            row[f"{metric}_std"] = stats.get(f"{metric}_std", np.nan)
            row[f"{metric}_min"] = stats.get(f"{metric}_min", np.nan)
            row[f"{metric}_max"] = stats.get(f"{metric}_max", np.nan)
            row[f"{metric}_ci_lower"] = stats.get(f"{metric}_ci_lower", np.nan)
            row[f"{metric}_ci_upper"] = stats.get(f"{metric}_ci_upper", np.nan)
        
        # Extract evaluation metrics statistics
        for key, value in stats.items():
            if key.startswith("metric_") and key.endswith("_mean"):
                metric_base = key[:-5]  # Remove "_mean" suffix
                row[key] = value
                row[f"{metric_base}_std"] = stats.get(f"{metric_base}_std", np.nan)
                row[f"{metric_base}_ci_lower"] = stats.get(f"{metric_base}_ci_lower", np.nan)
                row[f"{metric_base}_ci_upper"] = stats.get(f"{metric_base}_ci_upper", np.nan)
        
        # Extract configuration details
        row["latent_dim"] = stats.get("latent_dim", np.nan)
        row["num_layers"] = stats.get("num_layers", np.nan)
        row["channels_ratio"] = stats.get("channels_ratio", np.nan)
        
        rows.append(row)
    
    return pd.DataFrame(rows)

def get_dataframe(all_results: List[Dict], sensitivity_stats: List[Dict], enable_sensitivity_analysis: bool, sensitivity_n_runs: int) -> pd.DataFrame:
    """
    Create a DataFrame from all results and sensitivity statistics.
    
    Args:
        all_results: List of individual run results
        sensitivity_stats: List of aggregated sensitivity statistics
        enable_sensitivity_analysis: Whether sensitivity analysis is enabled
        sensitivity_n_runs: Number of runs per experiment for sensitivity analysis
        
    Returns:
        Combined DataFrame with all results and statistics
    """
    # Create results DataFrames
    if len(all_results) > 0:
        # Individual runs DataFrame (for detailed analysis)
        individual_results_df = create_results_dataframe(all_results)
        # Sensitivity statistics DataFrame (for statistical analysis)
        if enable_sensitivity_analysis and len(sensitivity_stats) > 0:
            results_df = create_sensitivity_dataframe(sensitivity_stats)
            print(f"📊 Sensitivity Analysis Results DataFrame created with {len(results_df)} experiments")
            print(f"   Each experiment represents statistics from {sensitivity_n_runs} runs")
            print(f"   Individual runs DataFrame: {len(individual_results_df)} total runs")
        else:
            results_df = individual_results_df
            print(f"📊 Results DataFrame created with {len(results_df)} experiments (single runs)")
        
        print(f"   Columns: {list(results_df.columns)}")
        
        # Save aggregated results
        if save_detailed_results:
            if enable_sensitivity_analysis and len(sensitivity_stats) > 0:
                # Save sensitivity statistics
                sensitivity_csv = results_path / "sensitivity_analysis_results.csv"
                results_df.to_csv(sensitivity_csv, index=False)
                print(f"   💾 Sensitivity analysis results saved to: {sensitivity_csv}")
                
                # Save individual runs
                individual_csv = results_path / "individual_runs_results.csv"
                individual_results_df.to_csv(individual_csv, index=False)
                print(f"   💾 Individual runs results saved to: {individual_csv}")
            else:
                results_csv = results_path / "aggregated_results.csv"
                results_df.to_csv(results_csv, index=False)
                print(f"   💾 Aggregated results saved to: {results_csv}")
        
        # Display basic statistics
        print(f"\n📈 Basic Statistics:")
        
        if enable_sensitivity_analysis and len(sensitivity_stats) > 0:
            print(f"\n   🔄 Sensitivity Analysis Summary:")
            print(f"     Experiments: {len(results_df)}")
            print(f"     Total runs: {len(individual_results_df)}")
            print(f"     Average success rate: {results_df['success_rate'].mean():.1%}")
            
            # Performance metrics summary with confidence intervals
            metric_cols = [col for col in results_df.columns if col.startswith("metric_") and col.endswith("_mean")]
            if metric_cols:
                print(f"\n   📊 Performance Metrics (with 95% Confidence Intervals):")
                for col in metric_cols:
                    metric_name = col.replace("metric_", "").replace("_mean", "")
                    std_col = col.replace("_mean", "_std")
                    ci_lower_col = col.replace("_mean", "_ci_lower")
                    ci_upper_col = col.replace("_mean", "_ci_upper")
                    
                    if results_df[col].notna().any():
                        mean_val = results_df[col].mean()
                        avg_std = results_df[std_col].mean() if std_col in results_df.columns else 0
                        
                        # Get average CI width if available
                        if ci_lower_col in results_df.columns and ci_upper_col in results_df.columns:
                            valid_ci = results_df[[ci_lower_col, ci_upper_col]].dropna()
                            if len(valid_ci) > 0:
                                avg_ci_lower = valid_ci[ci_lower_col].mean()
                                avg_ci_upper = valid_ci[ci_upper_col].mean()
                                print(f"     {metric_name:20}: {mean_val:.6f} ± {avg_std:.6f} (avg 95% CI: [{avg_ci_lower:.6f}, {avg_ci_upper:.6f}])")
                            else:
                                print(f"     {metric_name:20}: {mean_val:.6f} ± {avg_std:.6f}")
                        else:
                            print(f"     {metric_name:20}: {mean_val:.6f} ± {avg_std:.6f}")
            
            # Training metrics summary with confidence intervals
            print(f"\n   🎯 Training Metrics (with 95% Confidence Intervals):")
            train_cols = ["final_train_loss_mean", "final_val_loss_mean", "best_val_loss_mean"]
            for col in train_cols:
                if col in results_df.columns and results_df[col].notna().any():
                    std_col = col.replace("_mean", "_std")
                    ci_lower_col = col.replace("_mean", "_ci_lower")
                    ci_upper_col = col.replace("_mean", "_ci_upper")
                    
                    mean_val = results_df[col].mean()
                    avg_std = results_df[std_col].mean() if std_col in results_df.columns else 0
                    
                    metric_name = col.replace("_mean", "")
                    if ci_lower_col in results_df.columns and ci_upper_col in results_df.columns:
                        valid_ci = results_df[[ci_lower_col, ci_upper_col]].dropna()
                        if len(valid_ci) > 0:
                            avg_ci_lower = valid_ci[ci_lower_col].mean()
                            avg_ci_upper = valid_ci[ci_upper_col].mean()
                            print(f"     {metric_name:20}: {mean_val:.6f} ± {avg_std:.6f} (avg 95% CI: [{avg_ci_lower:.6f}, {avg_ci_upper:.6f}])")
                        else:
                            print(f"     {metric_name:20}: {mean_val:.6f} ± {avg_std:.6f}")
                    else:
                        print(f"     {metric_name:20}: {mean_val:.6f} ± {avg_std:.6f}")
        else:
            # Standard analysis for single runs
            # Performance metrics summary
            metric_cols = [col for col in results_df.columns if col.startswith("metric_")]
            if metric_cols:
                print(f"\n   Performance Metrics:")
                for col in metric_cols:
                    metric_name = col.replace("metric_", "")
                    if results_df[col].notna().any():
                        mean_val = results_df[col].mean()
                        std_val = results_df[col].std()
                        min_val = results_df[col].min()
                        max_val = results_df[col].max()
                        print(f"     {metric_name:8}: {mean_val:.6f} ± {std_val:.6f} (range: {min_val:.6f} - {max_val:.6f})")
            
            # Training metrics summary
            print(f"\n   Training Metrics:")
            train_cols = ["final_train_loss", "final_val_loss", "best_val_loss"]
            for col in train_cols:
                if col in results_df.columns and results_df[col].notna().any():
                    mean_val = results_df[col].mean()
                    std_val = results_df[col].std()
                    print(f"     {col:18}: {mean_val:.6f} ± {std_val:.6f}")
        
        # Model complexity summary
        print(f"\n   🏗️  Model Complexity:")
        if enable_sensitivity_analysis and "total_parameters_mean" in results_df.columns:
            mean_params = results_df["total_parameters_mean"].mean()
            std_params = results_df["total_parameters_std"].mean() if "total_parameters_std" in results_df.columns else 0
            print(f"     Total parameters:   {mean_params:,.0f} ± {std_params:,.0f}")
        elif "total_parameters" in results_df.columns:
            mean_params = results_df["total_parameters"].mean()
            std_params = results_df["total_parameters"].std()
            print(f"     Total parameters:   {mean_params:,.0f} ± {std_params:,.0f}")
        
        # Timing summary
        print(f"\n   ⏱️  Timing:")
        if enable_sensitivity_analysis and "duration_seconds_mean" in results_df.columns:
            mean_duration = results_df["duration_seconds_mean"].mean()
            std_duration = results_df["duration_seconds_std"].mean() if "duration_seconds_std" in results_df.columns else 0
            print(f"     Duration per experiment: {mean_duration:.1f} ± {std_duration:.1f} seconds")
        elif "duration_seconds" in results_df.columns:
            mean_duration = results_df["duration_seconds"].mean()
            std_duration = results_df["duration_seconds"].std()
            print(f"     Duration per experiment: {mean_duration:.1f} ± {std_duration:.1f} seconds")
        
        # Display first few rows
        print(f"\n📋 Sample Results (first 3 rows):")
        if enable_sensitivity_analysis:
            display_cols = ["experiment_name", "layer_mode", "model_class", "success_rate", "final_val_loss_mean"]
            metric_mean_cols = [col for col in results_df.columns if col.startswith("metric_") and col.endswith("_mean")][:2]
            display_cols.extend(metric_mean_cols)
        else:
            display_cols = ["experiment_name", "layer_mode", "model_class", "final_val_loss"]
            metric_cols = [col for col in results_df.columns if col.startswith("metric_")][:2]
            display_cols.extend(metric_cols)
        
        display_cols = [col for col in display_cols if col in results_df.columns]
        
        if len(results_df) > 0:
            display(results_df[display_cols].head(3))
        
        if enable_sensitivity_analysis:
            print(f"\n✅ Sensitivity analysis completed!")
            print(f"🔄 Each experiment provides mean ± std and 95% confidence intervals")
        else:
            print(f"\n✅ Results analysis completed!")
    
    else:
        print("❌ No results to analyze - all experiments failed")
    return results_df, individual_results_df

## Sensitivity Analysis Visualization

### 🔄 Statistical Analysis Across Multiple Runs

This section provides dedicated visualization and analysis of the sensitivity analysis results. When experiments are run multiple times, we can assess:

- **Stability & Reproducibility**: How consistent are the results across different random seeds?
- **Confidence Intervals**: Statistical confidence in our measurements
- **Variability Patterns**: Which configurations are more or less stable?
- **Statistical Significance**: Are observed differences statistically meaningful?

The visualizations include error bars representing standard deviation and confidence intervals where applicable.

In [ ]:
def sensitivity_visualization(results_df: pd.DataFrame, sensitivity_stats: List[Dict[str, Any]], individual_results_df: pd.DataFrame):
    # Sensitivity Analysis Visualization and Statistical Analysis
    if enable_sensitivity_analysis and len(sensitivity_stats) > 0 and len(individual_results_df) > 0:
        print("🔄 Creating Sensitivity Analysis Visualizations...")
        
        # Create comprehensive sensitivity analysis visualization
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes = axes.flatten()
        
        # 1. Success Rate Analysis
        ax = axes[0]
        if "success_rate" in results_df.columns:
            sns.barplot(data=results_df, x="experiment_name", y="success_rate", ax=ax, color='green')
            ax.set_title("Experiment Success Rate", fontweight='bold')
            ax.set_ylabel("Success Rate")
            ax.tick_params(axis='x', rotation=45)
            ax.set_ylim(0, 1.1)
            
            # Add percentage labels
            for i, v in enumerate(results_df["success_rate"]):
                ax.text(i, v + 0.02, f'{v:.1%}', ha='center', va='bottom')
        
        # 2. Validation Loss Variability
        ax = axes[1]
        if "final_val_loss_mean" in results_df.columns and "final_val_loss_std" in results_df.columns:
            # Create error bar plot
            x_pos = range(len(results_df))
            ax.errorbar(x_pos, results_df["final_val_loss_mean"], 
                    yerr=results_df["final_val_loss_std"], 
                    fmt='o', capsize=5, capthick=2, markersize=8)
            ax.set_xticks(x_pos)
            ax.set_xticklabels(results_df["experiment_name"], rotation=45)
            ax.set_title("Validation Loss: Mean ± Std", fontweight='bold')
            ax.set_ylabel("Final Validation Loss")
            ax.grid(True, alpha=0.3)
        
        # 3. Coefficient of Variation (CV) Analysis
        ax = axes[2]
        if "final_val_loss_mean" in results_df.columns and "final_val_loss_std" in results_df.columns:
            # Calculate coefficient of variation (std/mean) - shows relative variability
            cv = results_df["final_val_loss_std"] / results_df["final_val_loss_mean"]
            results_df_with_cv = results_df.copy()
            results_df_with_cv['cv'] = cv
            
            sns.barplot(data=results_df_with_cv, x="experiment_name", y="cv", ax=ax, color='orange')
            ax.set_title("Coefficient of Variation (CV)", fontweight='bold')
            ax.set_ylabel("CV (std/mean)")
            ax.tick_params(axis='x', rotation=45)
            
            # Add CV values as labels
            for i, v in enumerate(cv):
                ax.text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom')
        
        # 4. Training Time Variability
        ax = axes[3]
        if "duration_seconds_mean" in results_df.columns and "duration_seconds_std" in results_df.columns:
            x_pos = range(len(results_df))
            ax.errorbar(x_pos, results_df["duration_seconds_mean"], 
                    yerr=results_df["duration_seconds_std"], 
                    fmt='s', capsize=5, capthick=2, markersize=8, color='purple')
            ax.set_xticks(x_pos)
            ax.set_xticklabels(results_df["experiment_name"], rotation=45)
            ax.set_title("Training Time: Mean ± Std", fontweight='bold')
            ax.set_ylabel("Duration (seconds)")
            ax.grid(True, alpha=0.3)
        
        # 5.Circular Shift Consistency Box Plot
        ax = axes[4]
        # Create box plot from individual runs
        circular_shift_by_exp = []
        experiment_names = []
        
        for exp_name in results_df["experiment_name"].unique():
            exp_runs = individual_results_df[individual_results_df["experiment_name"] == exp_name]
            if len(exp_runs) > 0 and "metric_circular_shift_consistency" in exp_runs.columns:
                losses = exp_runs["metric_circular_shift_consistency"].dropna().tolist()
                circular_shift_by_exp.extend(losses)
                experiment_names.extend([exp_name] * len(losses))
        
        if circular_shift_by_exp:
            box_data = pd.DataFrame({
                'experiment': experiment_names,
                'circular_shift': circular_shift_by_exp
            })
            sns.boxplot(data=box_data, x="experiment", y="circular_shift", ax=ax)
            ax.set_title("Circular Shift Consistency Distribution", fontweight='bold')
            ax.set_ylabel("Final Circular Shift Consistency")
            ax.tick_params(axis='x', rotation=45)
        
        # 6. Stability Ranking
        ax = axes[5]
        if "final_val_loss_std" in results_df.columns:
            # Sort by standard deviation (lower = more stable)
            stability_ranking = results_df.sort_values("final_val_loss_std")
            
            sns.barplot(data=stability_ranking, x="experiment_name", y="final_val_loss_std", ax=ax, color='red')
            ax.set_title("Stability Ranking (Lower = More Stable)", fontweight='bold')
            ax.set_ylabel("Standard Deviation")
            ax.tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.suptitle("Sensitivity Analysis: Statistical Properties Across Multiple Runs", fontsize=16, fontweight='bold', y=1.02)
        plt.show()
        
        # Statistical Analysis Summary
        print(f"\n📊 Statistical Analysis Summary:")
        print(f"   Total experiments: {len(results_df)}")
        print(f"   Runs per experiment: {sensitivity_n_runs}")
        print(f"   Total individual runs: {len(individual_results_df)}")
        
        # Overall success rate
        overall_success_rate = results_df["success_rate"].mean()
        print(f"   Overall success rate: {overall_success_rate:.1%}")
        
        # Stability analysis
        if "final_val_loss_std" in results_df.columns:
            most_stable_exp = results_df.loc[results_df["final_val_loss_std"].idxmin()]
            least_stable_exp = results_df.loc[results_df["final_val_loss_std"].idxmax()]
            
            print(f"\n🏆 Most Stable Configuration:")
            print(f"   Experiment: {most_stable_exp['experiment_name']}")
            print(f"   Std Dev: {most_stable_exp['final_val_loss_std']:.6f}")
            print(f"   Mean Loss: {most_stable_exp['final_val_loss_mean']:.6f}")
            
            print(f"\n⚠️  Least Stable Configuration:")
            print(f"   Experiment: {least_stable_exp['experiment_name']}")
            print(f"   Std Dev: {least_stable_exp['final_val_loss_std']:.6f}")
            print(f"   Mean Loss: {least_stable_exp['final_val_loss_mean']:.6f}")
            
            # Calculate coefficient of variation for all experiments
            cv_values = results_df["final_val_loss_std"] / results_df["final_val_loss_mean"]
            avg_cv = cv_values.mean()
            print(f"\n📈 Average Coefficient of Variation: {avg_cv:.3f}")
            print(f"   Interpretation: {'Low variability' if avg_cv < 0.1 else 'Moderate variability' if avg_cv < 0.2 else 'High variability'}")
        
        # Confidence interval analysis
        print(f"\n🎯 Confidence Interval Analysis (95% CI):")
        if "final_val_loss_ci_lower" in results_df.columns and "final_val_loss_ci_upper" in results_df.columns:
            for _, row in results_df.iterrows():
                exp_name = row["experiment_name"]
                mean_loss = row["final_val_loss_mean"]
                ci_lower = row["final_val_loss_ci_lower"]
                ci_upper = row["final_val_loss_ci_upper"]
                ci_width = ci_upper - ci_lower
                
                print(f"   {exp_name:25}: {mean_loss:.6f} [{ci_lower:.6f}, {ci_upper:.6f}] (width: {ci_width:.6f})")
        
        # Statistical significance tests (if multiple experiments)
        if len(results_df) >= 2:
            print(f"\n🧪 Statistical Significance Analysis:")
            
            # Perform pairwise t-tests between experiments
            from scipy.stats import ttest_ind
            
            exp_names = results_df["experiment_name"].tolist()
            significant_pairs = []
            
            for i in range(len(exp_names)):
                for j in range(i+1, len(exp_names)):
                    exp1_name = exp_names[i]
                    exp2_name = exp_names[j]
                    
                    # Get individual run results for both experiments
                    exp1_runs = individual_results_df[individual_results_df["experiment_name"] == exp1_name]["final_val_loss"].dropna()
                    exp2_runs = individual_results_df[individual_results_df["experiment_name"] == exp2_name]["final_val_loss"].dropna()
                    
                    if len(exp1_runs) >= 2 and len(exp2_runs) >= 2:
                        t_stat, p_value = ttest_ind(exp1_runs, exp2_runs)
                        
                        if p_value < 0.05:
                            significant_pairs.append((exp1_name, exp2_name, p_value))
                            print(f"   Significant difference between {exp1_name} and {exp2_name} (p={p_value:.4f})")
            
            if not significant_pairs:
                print(f"   No statistically significant differences found (α = 0.05)")
        
        print(f"\n✅ Sensitivity analysis visualization completed!")

    elif enable_sensitivity_analysis:
        print("⚠️  Sensitivity analysis enabled but no results available")
    else:
        print("ℹ️  Sensitivity analysis disabled - showing single run results only")

## Reconstruction Model Experiments

### 🔄 Evaluating Downsampling Impact on Image Reconstruction

This section analyzes how different downsampling strategies affect reconstruction model performance. We evaluate AutoEncoder architectures with various downsampling approaches.

### 📊 Reconstruction-Specific Metrics

- **MSE (Mean Squared Error)**: Pixel-wise reconstruction accuracy
- **PSNR (Peak Signal-to-Noise Ratio)**: Image quality measure in dB
- **SSIM (Structural Similarity Index)**: Perceptual similarity measure

### 📊 Shift-Equivariance-Specific Metrics

- **Cr. S (Circular Shift Consistency)**:


### 🎯 Key Research Questions for Reconstruction

1. **How do different downsampling strategies affect reconstruction quality?**
2. **How do different downsampling strategies affect shift-equivariance related metrics?**

In [ ]:
reconstruction_complex = run_batch_experiments(reconstruction_experiments)

In [ ]:
reconstruction_real = run_batch_experiments(reconstruction_real_experiments)

In [ ]:
results_df_reconstruction_complex, individual_results_df_reconstruction_complex = get_dataframe(reconstruction_complex[0], reconstruction_complex[1], enable_sensitivity_analysis, sensitivity_n_runs)
results_df_reconstruction_real, individual_results_df_reconstruction_real = get_dataframe(reconstruction_real[0], reconstruction_real[1], enable_sensitivity_analysis, sensitivity_n_runs)
individual_results_df_reconstruction = pd.concat([individual_results_df_reconstruction_complex, individual_results_df_reconstruction_real], ignore_index=True)
results_df_reconstruction = pd.concat([results_df_reconstruction_complex, results_df_reconstruction_real], ignore_index=True)

In [ ]:
individual_sub_df_reconstruction = individual_results_df_reconstruction[["experiment_name", "total_parameters", "trainable_parameters", "metric_mse", "metric_psnr", "metric_h_alpha_metrics_accuracy", "metric_h_alpha_metrics_f1_macro",   "metric_cameron_metrics_accuracy", "metric_cameron_metrics_f1_macro", "metric_circular_shift_consistency"]].copy()
individual_sub_df_reconstruction.sort_values(by=["metric_mse"], inplace=True)
individual_sub_df_reconstruction.reset_index(drop=True, inplace=True)
sub_df_reconstruction = results_df_reconstruction[["experiment_name", "total_parameters", "trainable_parameters", "metric_mse", "metric_psnr", "metric_h_alpha_metrics_accuracy", "metric_h_alpha_metrics_f1_macro",   "metric_cameron_metrics_accuracy", "metric_cameron_metrics_f1_macro", "metric_circular_shift_consistency"]].copy()
sub_df_reconstruction.sort_values(by=["metric_mse"], inplace=True)
sub_df_reconstruction.reset_index(drop=True, inplace=True)

In [ ]:
individual_sub_df_reconstruction

In [ ]:
sub_df_reconstruction

In [ ]:
sensitivity_visualization(results_df_reconstruction_complex, reconstruction_complex[1], individual_sub_df_reconstruction)

## Segmentation Model Experiments

### 🔄 Evaluating Downsampling Impact on Semantic Segmentation

This section analyzes how different downsampling strategies affect segmentation model performance. We evaluate UNet architectures with various downsampling approaches.

### 📊 Segmentation-Specific Metrics

- **Acc (Accuracy)**:
- **IoU (Intersection over Union)**:
- **F1 Score ()**:

### 📊 Shift-Equivariance-Specific Metrics

- **Cr. S (Circular Shift Consistency)**:

### 🎯 Key Research Questions for Segmentation

1. **How do different downsampling strategies affect segmentation quality?**
2. **How do different downsampling strategies affect shift-equivariance related metrics?**

In [ ]:
segmentation_complex = run_batch_experiments(segmentation_experiments)

In [ ]:
segmentation_real = run_batch_experiments(segmentation_real_experiments)

In [ ]:
results_df_segmentation_complex, individual_results_df_segmentation_complex = get_dataframe(segmentation_complex[0], segmentation_complex[1], enable_sensitivity_analysis, sensitivity_n_runs)
results_df_segmentation_real, individual_results_df_segmentation_real = get_dataframe(segmentation_real[0], segmentation_real[1], enable_sensitivity_analysis, sensitivity_n_runs)
individual_results_df_segmentation = pd.concat([individual_results_df_segmentation_complex, individual_results_df_segmentation_real], ignore_index=True)
results_df_segmentation = pd.concat([results_df_segmentation_complex, results_df_segmentation_real], ignore_index=True)

In [ ]:
individual_sub_df_segmentation = individual_results_df_segmentation[["experiment_name", "total_parameters", "trainable_parameters", "duration_seconds", "metric_weighted_accuracy", "metric_mean_iou", "metric_f1_macro"]].copy()
individual_sub_df_segmentation.sort_values(by=["metric_f1_macro"], inplace=True)
individual_sub_df_segmentation.reset_index(drop=True, inplace=True)
sub_df_segmentation = results_df_segmentation[["experiment_name", "total_parameters", "trainable_parameters", "duration_seconds", "metric_weighted_accuracy", "metric_mean_iou", "metric_f1_macro"]].copy()
sub_df_segmentation.sort_values(by=["metric_f1_macro"], inplace=True)
sub_df_segmentation.reset_index(drop=True, inplace=True)

In [ ]:
individual_sub_df_segmentation

In [ ]:
sub_df_segmentation

In [ ]:
sensitivity_visualization(results_df_segmentation_complex, reconstruction_complex[1], individual_sub_df_segmentation)